---
title: "Aggregate Module: Spatial Aggregation to Healthsheds"
execute:
    freeze: auto
engine: jupyter
---

## aggregate

> This module aggregates the downloaded data into the respective output dataframes.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

We prototyped the code in this module using a Jupyter notebook. The notebook is available in `notes/prototypes/learning_aggregations_w_michelle_20250328.ipynb`. The code in this module is a cleaned-up version of the code in that notebook. The notebook contains additional comments and explanations of the code, which may be helpful for understanding the code in this module.

The basic process is as follows:

1. Load the netCDF data in memory
2. Statistically aggregate the hourly data to daily data per exposure using resample()
3. Write out the data to tiff
4. Read the tiff data back in
5. Read in the shapefile that defines the healthsheds
6. Spatially aggregate the exposure data to the healthsheds
7. Quality check the aggregations
8. Write out final aggregations to tiff

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
import tempfile
import rasterio
import hydra
import argparse
import os

import pandas as pd
import geopandas as gpd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from dataclasses import dataclass, field
from typing import Optional, Tuple
from pyprojroot import here
from hydra import initialize, compose
from omegaconf import OmegaConf, DictConfig
from tqdm import tqdm
from math import ceil, floor
from rasterstats.io import Raster
from rasterstats.utils import boxify_points, rasterize_geom

try: from era5_sandbox.core import GoogleDriver, _get_callable, describe, ClimateDataFileHandler, kelvin_to_celsius
except: from core import GoogleDriver, _get_callable, describe, ClimateDataFileHandler, kelvin_to_celsius

In [ ]:
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

We're going to write a function that aggregates the data for a single exposure from a file. This file should be the single month data we got from the previous step in the pipeline.

In [ ]:
eg_file = here() / "bld/2009_01_nepal.nc"

In [0]:
#| echo: false
#| output: asis
show_doc(resample_netcdf)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L36){target="_blank" style="float:right; font-size:smaller"}

### resample_netcdf

>      resample_netcdf (fpath:str, resample:str='1D', agg_func:<built-
>                       infunctioncallable>=<function mean at 0x14b8f1d4fbf0>,
>                       time_dim:str='valid_time', **xr_open_kwargs)

*Resample a netCDF file to a specified frequency and aggregation method.

Args:
    fpath (str): Path to the netCDF file.
    resample (str): Resampling frequency (e.g., '1H', '1D').
    agg_func (callable): Aggregation function (e.g., np.mean, np.sum).

Returns:
    xarray.Dataset: Resampled dataset.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| fpath | str |  | Path to the netCDF file. |
| resample | str | 1D | Resampling frequency (e.g., '1H', '1D') |
| agg_func | callable | mean | Aggregation function (e.g., np.mean, np.sum). |
| time_dim | str | valid_time | Name of the time dimension in the dataset. |
| xr_open_kwargs | VAR_KEYWORD |  |  |
| **Returns** | **Dataset** |  | **keywords for python's xarray module** |

We pull the aggregation function from the config file:

In [ ]:
var = 'swvl1'
agg_func = _get_callable(cfg['aggregation']['aggregation'][var]['hourly_to_daily'][0]['function'])

In [ ]:
with ClimateDataFileHandler(eg_file) as handler:

    ds_path = handler.get_dataset("instant")
    resampled_data = resample_netcdf(ds_path, agg_func=agg_func)

I'm going to use a dataclass to represent the tiff data. This will allow us to easily pass around the data and metadata associated with the tiff file. Why? I've never used dataclasses and I'm curious about them — ChatGPT thinks this will make the code cleaner and easier to read.

In [0]:
#| echo: false
#| output: asis
show_doc(RasterFile)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L66){target="_blank" style="float:right; font-size:smaller"}

### RasterFile

>      RasterFile (path:str, band:int)

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
@dataclass
class RasterFile:
    path: str
    band: int # note that this is 1-indexed
    data: Optional[np.ndarray] = field(default=None, init=False)
    transform: Optional[rasterio.Affine] = field(default=None, init=False)
    crs: Optional[str] = field(default=None, init=False)
    nodata: Optional[float] = field(default=None, init=False)
    bounds: Optional[Tuple[float, float, float, float]] = field(default=None, init=False)

    def load(self):
        """Load raster data and basic metadata."""
        with rasterio.open(self.path) as src:
            self.data = src.read(self.band)  # each day gets one rasterfile
            self.transform = src.transform
            self.crs = src.crs
            self.nodata = src.nodata
            self.bounds = src.bounds
        return self

    def shape(self) -> Optional[Tuple[int, int]]:
        """Return the shape of the raster data."""
        return self.data.shape if self.data is not None else None

    def __str__(self):
        return f"RasterFile(path='{self.path}', shape={self.shape()}, crs='{self.crs}')"

Next, a function to write and read the netCDF to tiff:

In [0]:
#| echo: false
#| output: asis
show_doc(netcdf_to_tiff)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L94){target="_blank" style="float:right; font-size:smaller"}

### netcdf_to_tiff

>      netcdf_to_tiff (ds:xarray.core.dataset.Dataset, band:int, variable:str,
>                      crs:str='EPSG:4326')

*Convert a netCDF file to a GeoTIFF file.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ds | Dataset |  | The aggregated xarray dataset to convert. |
| band | int |  | The day to rasterise; 1 indexed just like human english |
| variable | str |  | The variable name to convert. |
| crs | str | EPSG:4326 | Coordinate reference system (default is WGS84). |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def netcdf_to_tiff(
    ds: xr.Dataset, # The aggregated xarray dataset to convert.    
    band: int,      # The day to rasterise; 1 indexed just like human english
    variable: str, # The variable name to convert.
    crs: str = "EPSG:4326", # Coordinate reference system (default is WGS84).    
    ):

    """
    Convert a netCDF file to a GeoTIFF file.
    """

    with tempfile.TemporaryDirectory() as tmpdirname:

        # Select the variable and time index
        variable = ds[variable]
        ds_ = variable.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
        ds_ = ds_.rio.write_crs(crs)
        # Save as GeoTIFF
        ds_.rio.to_raster(f"{tmpdirname}/output.tif")
        # Load the raster file
        raster_file = RasterFile(path=f"{tmpdirname}/output.tif", band=band).load()

    return raster_file

Now to test it:

In [ ]:
with ClimateDataFileHandler(eg_file) as handler:
    ds_path = handler.get_dataset("instant")
    resampled_nc = resample_netcdf(ds_path)

print(resampled_nc)
resampled_tiff = netcdf_to_tiff(
    ds=resampled_nc,
    band=28,
    variable="swvl1",
    crs="EPSG:4326"
)

In [ ]:
resampled_tiff.data.shape, resampled_tiff.transform, resampled_tiff.crs, resampled_tiff.bounds

Super cool! The tiff file is created and the data is read back in correctly. Now we can move on to the next step, which is to aggregate the data by healthshed.

## Polygon to Raster Cells

This function was initially shared from a previous NSAPH aggregation pipeline [here](https://github.com/NSAPH-Data-Processing/air_pollution__aqdh/blob/2a8109075fe7a8fbf7c435cc34ffa97b63f5e133/utils/faster_zonal_stats.py#L17). To better understand this, here is a ChatGPT explanation of the code:

> This function, [`polygon_to_raster_cells`](https://TinasheMTapera.github.io/era5_sandbox/aggregate.html#polygon_to_raster_cells), is doing a crucial first step in spatial alignment: it determines which raster cells are “touched” by each polygon geometry (e.g., administrative areas, watersheds, etc.).    
Essentially, this function helps figure out which pixels from a raster image fall inside each polygon (like a district, region, or shape). It does this by looking at each polygon one by one, zooming in on just the part of the raster that overlaps with that shape, and marking the pixels that are inside. This is kind of like placing a cookie cutter (the polygon) on a pixelated map (the raster) and seeing which pixels get cut.  
The result is a list where each item tells you the pixel locations that match a specific polygon. You can then use those pixel locations to pull out data from the raster, like temperatures or rainfall, and calculate statistics (like the average) for each shape. This is a key step when you want to summarize raster data within specific regions, like figuring out the average temperature in each county or how much vegetation is in each park.

In [0]:
#| echo: false
#| output: asis
show_doc(polygon_to_raster_cells)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L120){target="_blank" style="float:right; font-size:smaller"}

### polygon_to_raster_cells

>      polygon_to_raster_cells (vectors, raster, nodata=None, affine=None,
>                               all_touched=False, verbose=False, **kwargs)

*Returns an index map for each vector geometry to indices in the raster source.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| vectors |  |  | list of geometries from a shapefile |
| raster |  |  | the raster data as a numpy array |
| nodata | NoneType | None | the nodata value of the raster |
| affine | NoneType | None | the affine transform of the raster |
| all_touched | bool | False | whether to include all touched pixels |
| verbose | bool | False |  |
| kwargs | VAR_KEYWORD |  |  |
| **Returns** | **list** |  | **A dictionary mapping vector the ids of geometries to locations (indices) in the raster source.** |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def polygon_to_raster_cells(
    vectors, # list of geometries from a shapefile
    raster, # the raster data as a numpy array
    nodata=None, # the nodata value of the raster
    affine=None, # the affine transform of the raster
    all_touched=False, # whether to include all touched pixels
    verbose=False, 
    **kwargs,
) -> list: # A dictionary mapping vector the ids of geometries to locations (indices) in the raster source.
    """Returns an index map for each vector geometry to indices in the raster source."""

    cell_map = []

    with Raster(raster, affine, nodata) as rast:
        # used later to crop raster and find start row and col
        min_lon, dlon = affine.c, affine.a
        max_lat, dlat = affine.f, -affine.e
        H, W = rast.shape

        for geom in tqdm(vectors, disable=(not verbose)):
            if "Point" in geom.geom_type:
                geom = boxify_points(geom, rast)

            # find geometry bounds to crop raster
            # the raster and geometry must be in the same lon/lat coordinate system
            start_row = max(0, min(H - 1, floor((max_lat - geom.bounds[3]) / dlat)))
            start_col = min(W - 1, max(0, floor((geom.bounds[0] - min_lon) / dlon)))
            end_col = max(0, min(W - 1, ceil((geom.bounds[2] - min_lon) / dlon)))
            end_row = min(H - 1, max(0, ceil((max_lat - geom.bounds[1]) / dlat)))
            geom_bounds = (
                min_lon + dlon * start_col,  # left
                max_lat - dlat * end_row - 1e-12,  # bottom
                min_lon + dlon * end_col + 1e-12,  # right
                max_lat - dlat * start_row,  # top
            )

            # crop raster to area of interest and rasterize
            fsrc = rast.read(bounds=geom_bounds)
            rv_array = rasterize_geom(geom, like=fsrc, all_touched=all_touched)
            indices = np.nonzero(rv_array)

            if len(indices[0]) > 0:
                indices = (indices[0] + start_row, indices[1] + start_col)
                assert 0 <= indices[0].min() < rast.shape[0]
                assert 0 <= indices[1].min() < rast.shape[1]
            else:
                pass  # stop here for debug

            cell_map.append(indices)

        return cell_map

To use this, we must define the polygon and raster data. The polygon data is the healthshed shapefile, and the raster data is the tiff file we created earlier. We can use the [`GoogleDriver`](https://TinasheMTapera.github.io/era5_sandbox/core.html#googledriver) class we defined in `core` to read in the shapefile.

In [ ]:
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = driver.get_drive()
healthsheds = driver.read_healthsheds("Nepal_Healthsheds2024.zip")

In [ ]:
res_poly2cell=polygon_to_raster_cells(
    vectors = healthsheds.geometry.values, # the geometries of the shapefile of the regions
    raster=resampled_tiff.data, # the raster data above
    nodata=resampled_tiff.nodata, # any intersections with no data, may have to be np.nan
    affine=resampled_tiff.transform, # some math thing need to revise
    all_touched=True, 
    verbose=True
)

The data below maps which grid entries fall into each of the regions in the shapefile (e.g. which pixel is in which state)

In [ ]:
res_poly2cell[:5]

Last but not least, we aggregate these data to the healthshed level. We can use the `rasterstats` package to do this.

In [0]:
#| echo: false
#| output: asis
show_doc(aggregate_to_healthsheds)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L174){target="_blank" style="float:right; font-size:smaller"}

### aggregate_to_healthsheds

>      aggregate_to_healthsheds (res_poly2cell:list, raster:__main__.RasterFile,
>                                shapes:geopandas.geodataframe.GeoDataFrame,
>                                names_column:str='fs_uid',
>                                aggregation_func:<built-
>                                infunctioncallable>=<function nanmean at
>                                0x14b8f1ccf8f0>, aggregation_name:str='mean')

*Aggregate the raster data to the health sheds.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| res_poly2cell | list |  | the result of polygon_to_raster_cells |
| raster | RasterFile |  | the raster data |
| shapes | GeoDataFrame |  | the shapes of the health sheds |
| names_column | str | fs_uid | the unique identifier column name of the health sheds |
| aggregation_func | callable | nanmean | the aggregation function |
| aggregation_name | str | mean | the name of the aggregation function |
| **Returns** | **GeoDataFrame** |  |  |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def aggregate_to_healthsheds(
    res_poly2cell: list, # the result of polygon_to_raster_cells    
    raster: RasterFile, # the raster data
    shapes: gpd.GeoDataFrame, # the shapes of the health sheds
    names_column: str = "fs_uid", # the unique identifier column name of the health sheds
    aggregation_func: callable = np.nanmean, # the aggregation function
    aggregation_name: str = "mean" # the name of the aggregation function
    ) -> gpd.GeoDataFrame:
    """
    Aggregate the raster data to the health sheds.
    """

    stats = []

    for indices in res_poly2cell:
        if len(indices[0]) == 0:
            # no cells found for this polygon
            stats.append(np.nan)
        else:
            cells = raster.data[indices]
            if sum(~np.isnan(cells)) == 0:
                # no valid cells found for this polygon
                stats.append(np.nan)
                continue
            else:
                # compute MEAN of valid cells
                # but this stat can be ANYTHING
                stats.append(aggregation_func(cells))

    # clean up the result into a dataframe
    stats = pd.Series(stats)
    shapes[aggregation_name] = stats
    df = pd.DataFrame(
            {"healthshed": shapes[names_column], aggregation_name: stats}
        )
    gdf = gpd.GeoDataFrame(df, geometry=shapes.geometry.values, crs=shapes.crs)
    return gdf

And now we apply it:

In [ ]:
result = aggregate_to_healthsheds(
    res_poly2cell=res_poly2cell,
    raster=resampled_tiff,
    shapes=healthsheds,
    names_column="fid",
    aggregation_func=np.nanmean,
    aggregation_name="mean_soil_moisture"
)
result.head()

And plot for QA:

In [ ]:
result.plot(column="mean_soil_moisture", legend=True)
plt.title("Mean Soil Moisture (m^3 m^-3) by Health Shed Nov 2017 day 1")
plt.show()

That looks great! The data is aggregated to the healthshed level, and we can see the differences in exposure across the healthsheds. We can also see that the data is not uniform across the healthsheds, which is what we expect.

## Tests and Main

Now we can wrap this up in a main function that will simply take in the input file and generate this output. We can also add some tests to make sure the data is aggregated correctly; tests will run automatically in this notebook.

In [ ]:
import random

In [ ]:
#| eval: false
# variables = ["t2m", "d2m"]
# years = ["20{:02d}".format(m) for m in range(9, 24)]
# months = [str(m) for m in range(1, 13)]
# aggregations = [
#     ("Mean", np.nanmean),
#     ("Max", np.nanmax),
#     ("Min", np.nanmin)
# ]

# exposure_variable = random.choice(variables)
# year = random.choice(years)
# month = random.choice(months)
# aggregation_str, agg_func = random.choice(aggregations)
# input_file = here() / "data/input/{}_{}.nc".format(year, month)

# with initialize(version_base=None, config_path="../conf"):
#     cfg = compose(config_name='config.yaml')

# driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
# drive = driver.get_drive()
# healthsheds = driver.read_healthsheds(cfg.GOOGLE_DRIVE_AUTH_JSON.healthsheds_id)

# with ClimateDataFileHandler(input_file) as handler:
#     ds_path = handler.get_dataset("instant")
#     resampled_nc_file = resample_netcdf(ds_path, agg_func=agg_func)

# days = len(resampled_nc_file.valid_time.values)
# day = random.choice(range(1, days + 1))

# resampled_tiff = netcdf_to_tiff(
#     ds=resampled_nc_file,
#     band=day, # the day we're aggregating
#     variable=exposure_variable,
#     crs="EPSG:4326"
# )

# res_poly2cell=polygon_to_raster_cells(
#     vectors = healthsheds.geometry.values, # the geometries of the shapefile of the regions
#     raster=resampled_tiff.data, # the raster data above
#     nodata=resampled_tiff.nodata, # any intersections with no data, may have to be np.nan
#     affine=resampled_tiff.transform, # some math thing need to revise
#     all_touched=True, 
#     verbose=True
# )

# result = aggregate_to_healthsheds(
#     res_poly2cell=res_poly2cell,
#     raster=resampled_tiff,
#     shapes=healthsheds,
#     names_column="fs_uid",
#     aggregation_func=agg_func,
#     aggregation_name=exposure_variable
# )

# result.plot(column=exposure_variable, legend=True)
# plt.title("{} {} (K) by Health Shed {}".format(aggregation_str, exposure_variable, input_file.stem))
# plt.suptitle("Aggregation: {}, Day: {}".format(aggregation_str, str(day)))
# plt.show()

::: {.callout-note}
**Note:** The above code is commented out to prevent execution during documentation generation. You can uncomment and run it in an appropriate environment to test the aggregation process.
:::

3.2 seconds per aggregation is pretty cool!

In [ ]:
#| eval: false
result.to_parquet(here() / "data/testing/test_aggregation.parquet")

In [0]:
#| echo: false
#| output: asis
show_doc(aggregate_data)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L214){target="_blank" style="float:right; font-size:smaller"}

### aggregate_data

>      aggregate_data (cfg:omegaconf.dictconfig.DictConfig, input_file:str,
>                      output_file:str, exposure_variable:str)

*Aggregate raster data day-by-day and store all days and statistics as separate columns in a single Parquet file.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| cfg | DictConfig | the hydra config |
| input_file | str | the input netcdf file |
| output_file | str | the output parquet file |
| exposure_variable | str | Which variable in the dataset to aggregate |
| **Returns** | **None** |  |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def aggregate_data(
        cfg: DictConfig, # the hydra config
        input_file: str, # the input netcdf file
        output_file: str, # the output parquet file
        exposure_variable: str # Which variable in the dataset to aggregate
    ) -> None:
    '''
    Aggregate raster data day-by-day and store all days and statistics as separate columns in a single Parquet file.
    '''

    if cfg.development_mode:
        describe(cfg)
        return None

    geography = cfg['query'].geography
    year = cfg['query']['year']
    month = cfg['query']['month']
    daily_aggs = cfg['aggregation']['aggregation'][exposure_variable]['hourly_to_daily']
    healthshed_aggs = cfg['aggregation']['aggregation'][exposure_variable]['daily_to_healthshed']

    # Load healthsheds
    driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
    drive = driver.get_drive()
    healthsheds = driver.read_healthsheds(cfg.geographies[geography].healthsheds)
    
    # Initialize output DataFrame
    result_df = healthsheds[[cfg.geographies[geography].unique_id, "geometry"]].copy()

    for daily_agg in daily_aggs:
        print(f"Processing daily aggregation: {daily_agg['name']}...")
    
        daily_agg_func = _get_callable(daily_agg['function'])

        with ClimateDataFileHandler(input_file) as handler:
            if exposure_variable in ["t2m", "d2m", "swvl1"]:
                ds_path = handler.get_dataset("instant")
            else:
                ds_path = handler.get_dataset("accum")
            resampled_nc_file = resample_netcdf(ds_path, agg_func=daily_agg_func)
        
        for healthshed_agg in healthshed_aggs:
            print(f"Aggregating to healthshed by: {healthshed_agg['name']}...")

            # Get the number of days in the dataset
            days = len(resampled_nc_file.valid_time.values)

            # Get the aggregation function for healthshed
            healthshed_agg_func = _get_callable(healthshed_agg['function'])
            days = len(resampled_nc_file.valid_time.values)

            for day in range(1, days + 1):
                print(f"Processing day {day}...")
                
                day_col = f"day_{day:02d}_daily_{daily_agg['name']}"
                resampled_tiff = netcdf_to_tiff(
                    ds=resampled_nc_file,
                    band=day,
                    variable=exposure_variable,
                    crs="EPSG:4326"
                )

                result_poly2cell = polygon_to_raster_cells(
                    vectors=healthsheds.geometry.values,
                    raster=resampled_tiff.data,
                    nodata=resampled_tiff.nodata,
                    affine=resampled_tiff.transform,
                    all_touched=True,
                    verbose=True
                )

                res = aggregate_to_healthsheds(
                    res_poly2cell=result_poly2cell,
                    raster=resampled_tiff,
                    shapes=healthsheds,
                    names_column=cfg.geographies[geography].unique_id,
                    aggregation_func=healthshed_agg_func,
                    aggregation_name=exposure_variable
                )

                result_df[day_col] = res[exposure_variable]

    print(f"Saving final monthly parquet file: {output_file}")
    result_df.to_parquet(output_file, compression="snappy")
    # return(result_df)

In [ ]:
#| eval: false
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

cfg.development_mode = False
cfg.query['year'] = 2017
cfg.query['month'] = 11
cfg.query['geography'] = "nepal"

variable = "swvl1"

aggregate_data(cfg, here() / "bld/2017_11_nepal.nc", here() / "data/testing/test_nepal_aggregation.parquet", exposure_variable=variable)

In [ ]:
#| eval: false
parquet_file = gpd.read_parquet(here() / "data/testing/test_nepal_aggregation.parquet")

In [ ]:
#| eval: false
parquet_file

In [ ]:
#| eval: false
parquet_file.plot(column="day_22_daily_mean", legend=True)

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L302){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
@hydra.main(version_base=None, config_path="../../conf", config_name="config")
def main(cfg: DictConfig) -> None:
    # Parse command-line arguments
    input_file = str(snakemake.input[0])  # First input file
    output_file = str(snakemake.output[0])
    geography = str(snakemake.params.geography)
    aggregation_variable = str(snakemake.params.variable)

    variables_dict = {
        "2m_temperature": "t2m",
        "2m_dewpoint_temperature": "d2m",
        "volumetric_soil_water_layer_1": "swvl1",
        "total_precipitation": "tp"
    }

    cfg['query']['geography'] = geography
    
    aggregate_data(cfg, input_file=input_file, output_file=output_file, exposure_variable=variables_dict[aggregation_variable])